# Compute climate summaries

Derive daily UTC values for every grid cell with data, compute the ERA5-Land 1991–2020 reference climatology per cell, and build the monthly climate summaries: one row per site, study year and study month, plus the same summary for every cell in the region. Settings come from `configs/climate.yaml`; run `20_acquire_climate_data.ipynb` first. Results are saved under `outputs/climate/`.

In [1]:
import json
from pathlib import Path

import pandas as pd

from cds_pull import (
    ClimateConfig,
    build_monthly_summary,
    cell_table,
    compute_normals,
    load_daily,
    missing_days,
    open_region,
    run_checks,
)

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"

## Climate configuration

In [2]:
config = ClimateConfig.from_yaml(ROOT / "configs" / "climate.yaml", project_root=ROOT)
site_cells = pd.read_csv(config.output_dir / "site_cells.csv", keep_default_na=False)
region = open_region(config.output_dir / "raw" / "region")
cells = cell_table(region)
data_cells = cells.loc[cells["has_data"]]
config.to_dict()

{'sites_source': '/home/tshihadah/code/wadi-gaza-research/sites/StudySites.gpkg',
 'id_field': 'site_name',
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/climate',
 'dataset': 'reanalysis-era5-land-timeseries',
 'variables': ('2m_temperature', 'total_precipitation'),
 'region_lat': (31.2, 32.0),
 'region_lon': (34.2, 34.8),
 'site_cell_overrides': {'HolotPalmahim': 'N31.9_E34.8',
  'NahalRubin': 'N31.9_E34.8'},
 'extraction_range': ['1990-10-01', '2025-12-31'],
 'reference_period': (1991, 2020),
 'study_years': (2019, 2025),
 'study_months': (1, 5, 9, 11),
 'wet_season_months': (10, 4),
 'near_zero_normal_mm': 10,
 'min_reference_periods': 30,
 'daily_temp_range_c': (-5, 45),
 'daily_precip_cap_mm': 200}

## Daily values

Precipitation for day D sums the hourly values stamped D 01:00 through D+1 00:00. Temperature uses the values stamped D 00:00 through D 23:00, as a 24-hour mean and as a (max + min) / 2 midrange. A day without all 24 hourly values for a variable keeps its row and hour counts, but that variable's daily value is left blank.

In [3]:
daily = load_daily(region, data_cells, *config.extraction_range)
daily.to_csv(config.output_dir / "daily_climate.csv", index=False, float_format="%.4f")
print(f"{len(daily)} daily rows for {daily['cell_id'].nunique()} cells, "
      f"{daily['date'].min()} to {daily['date'].max()}")
daily.loc[daily["cell_id"].isin(site_cells["cell_id"])].groupby("cell_id")[
    ["precip_mm", "t2m_mean_24h_c", "t2m_midrange_c", "precip_clamped"]
].agg(["mean", "min", "max"]).round(2)

463536 daily rows for 36 cells, 1990-10-01 to 2025-12-31


precip_mm             t2m_mean_24h_c              t2m_midrange_c  \
                 mean  min    max           mean   min    max           mean   
cell_id                                                                        
N31.3_E34.2      0.49  0.0  30.29          20.76  7.09  33.88          20.89   
N31.3_E34.5      0.51  0.0  26.93          20.34  5.65  34.69          20.79   
N31.4_E34.5      0.67  0.0  30.96          20.44  6.02  33.77          20.86   
N31.5_E34.4      0.84  0.0  37.01          20.70  6.87  33.03          20.92   
N31.7_E34.6      1.04  0.0  43.21          20.11  5.77  32.35          20.34   
N31.9_E34.8      0.87  0.0  36.70          20.20  5.49  32.92          20.39   

                         precip_clamped          
              min    max           mean min max  
cell_id                                          
N31.3_E34.2  6.93  33.62           0.59   0   6  
N31.3_E34.5  5.86  34.12           0.57   0   6  
N31.4_E34.5  6.30  33.57           0.55   0   8  
N31.5_E34.4  7.18  33.47           0.54   0   7  
N31.7_E34.6  6.01  33.41           0.47   0   6  
N31.9_E34.8  5.66  33.36           0.50   0   6

## Missing days

Every day with fewer than 24 hourly values for either variable, in any cell. These are excluded from monthly sums and means, and any affected month or window is flagged. Review this table before using the summaries.

In [4]:
gaps = missing_days(daily)
gaps.to_csv(config.output_dir / "missing_days.csv", index=False)
print(f"{len(gaps)} incomplete day(s) across all cells.")
display(gaps)

0 incomplete day(s) across all cells.


,cell_id,date,precip_hours,t2m_hours


## Reference climatology

Normals per cell: monthly precipitation and temperature (calendar months 1991–2020), wet-season precipitation (seasons 1990/91 to 2019/20) and cumulative precipitation since October 1 for each study month (observation years 1991–2020).

In [5]:
normals = compute_normals(daily, config)
normals.to_csv(config.output_dir / "climate_normals.csv", index=False)
site_normals = normals.loc[normals["cell_id"].isin(site_cells["cell_id"])]
site_normals.loc[site_normals["month"].isin(config.study_months) | site_normals["month"].isna()].round(2)

,cell_id,measure,month,mean,sd,n_periods,period_first,period_last,incomplete_flag,near_zero_flag
287,N31.3_E34.2,cum_since_oct1_precip_mm,1,64.51,32.79,30,1991,2020,False,False
288,N31.3_E34.2,cum_since_oct1_precip_mm,5,175.49,45.15,30,1991,2020,False,False
289,N31.3_E34.2,cum_since_oct1_precip_mm,9,183.26,45.49,30,1991,2020,False,False
290,N31.3_E34.2,cum_since_oct1_precip_mm,11,12.91,13.18,30,1991,2020,False,False
291,N31.3_E34.2,precip_mm,1,44.28,20.73,30,1991,2020,False,False
...,...,...,...,...,...,...,...,...,...,...
1381,N31.9_E34.8,t2m_midrange_c,1,13.07,1.13,30,1991,2020,False,False
1385,N31.9_E34.8,t2m_midrange_c,5,21.58,1.06,30,1991,2020,False,False
1389,N31.9_E34.8,t2m_midrange_c,9,25.72,0.85,30,1991,2020,False,False
1391,N31.9_E34.8,t2m_midrange_c,11,19.26,1.20,30,1991,2020,False,False


## Monthly climate summaries

The site summary has 196 rows (seven sites × seven years × four study months). The regional summary repeats the same metrics for every cell with data.

In [6]:
summary = build_monthly_summary(daily, normals, site_cells, config)
summary.to_csv(config.output_dir / "monthly_climate_summary.csv", index=False)
regional = build_monthly_summary(daily, normals, data_cells.assign(site=data_cells["cell_id"]), config)
regional = regional.drop(columns="site")
regional.to_csv(config.output_dir / "regional_climate_summary.csv", index=False)
with (config.output_dir / "config.json").open("w", encoding="utf-8") as stream:
    json.dump(config.to_dict(), stream, indent=2)
print(f"Saved {len(summary)} site rows and {len(regional)} regional rows under {config.output_dir}")
summary_columns = [
    "site", "year", "month", "precip_mm", "precip_normal_mm", "precip_pct_normal",
    "wet_season_label", "wet_season_precip_mm", "wet_season_precip_pct_normal",
    "cum_since_oct1_precip_mm", "cum_since_oct1_precip_pct_normal",
    "t2m_mean_24h_anom_c", "t2m_midrange_anom_c", "month_incomplete",
]
summary.loc[summary["site"] == "WadiGaza", summary_columns].round(1)

Saved 196 site rows and 1008 regional rows under /home/tshihadah/code/wadi-gaza-research/outputs/climate


,site,year,month,precip_mm,precip_normal_mm,precip_pct_normal,wet_season_label,wet_season_precip_mm,wet_season_precip_pct_normal,cum_since_oct1_precip_mm,cum_since_oct1_precip_pct_normal,t2m_mean_24h_anom_c,t2m_midrange_anom_c,month_incomplete
168,WadiGaza,2019,1,32.0,81.3,39.3,2017/18,264.8,87.8,161.8,149.4,0.1,0.1,False
169,WadiGaza,2019,5,1.2,5.6,NaN,2018/19,367.3,121.8,367.3,121.8,0.8,1.0,False
170,WadiGaza,2019,9,2.0,2.9,NaN,2018/19,367.3,121.8,373.0,119.3,0.4,0.4,False
171,WadiGaza,2019,11,6.6,33.3,19.7,2018/19,367.3,121.8,36.1,217.9,1.5,1.6,False
172,WadiGaza,2020,1,122.6,81.3,150.8,2018/19,367.3,121.8,107.1,98.9,-0.3,-0.3,False
173,WadiGaza,2020,5,6.3,5.6,NaN,2019/20,384.6,127.5,384.6,127.5,0.9,0.9,False
174,WadiGaza,2020,9,2.3,2.9,NaN,2019/20,384.6,127.5,394.9,126.3,2.1,2.3,False
175,WadiGaza,2020,11,57.8,33.3,173.6,2019/20,384.6,127.5,2.5,15.0,0.3,0.2,False
176,WadiGaza,2021,1,64.7,81.3,79.5,2019/20,384.6,127.5,85.6,79.0,2.0,2.1,False
177,WadiGaza,2021,5,0.3,5.6,NaN,2020/21,238.3,79.0,238.3,79.0,1.7,1.8,False


## Automated checks and review flags

In [7]:
checks = run_checks(daily, normals, summary, site_cells, cells, config)
checks.to_csv(config.output_dir / "checks.csv", index=False)
display(checks)
failed = checks.loc[~checks["passed"]]
print(f"{len(failed)} check(s) failed.")
display(failed)
flagged = summary.loc[summary["month_incomplete"]
                      | summary["wet_season_missing_days"].gt(0)
                      | summary["cum_since_oct1_missing_days"].gt(0)]
print(f"{len(flagged)} summary row(s) affected by missing days.")
display(flagged[["site", "year", "month", "precip_missing_days", "t2m_missing_days",
                 "wet_season_missing_days", "cum_since_oct1_missing_days"]])

,check,scope,passed,detail
0,region_grid,all,True,"63 grid points, 36 with data"
1,site_cell_has_data,AlAhrashProtectorate,True,N31.3_E34.2 (centroid)
2,site_cell_has_data,BesorStream,True,N31.4_E34.5 (centroid)
3,site_cell_has_data,HaBesorNationalPark,True,N31.3_E34.5 (centroid)
4,site_cell_has_data,HolotNizzanim,True,N31.7_E34.6 (centroid)
...,...,...,...,...
151,negative_hours_clamped,N32.0_E34.8,True,6532 hourly values clamped to zero
152,normals_complete,all,True,0 of 1476 normals below 30 periods
153,summary_row_count,all,True,196 of 196 rows
154,one_cell_per_site,all,True,"7 sites, 6 cells"


0 check(s) failed.


,check,scope,passed,detail


0 summary row(s) affected by missing days.


,site,year,month,precip_missing_days,t2m_missing_days,wet_season_missing_days,cum_since_oct1_missing_days
